# 🤖 Sistema de Q&A com IA e Interface HTML Interativa

Este notebook implementa um sistema completo de perguntas e respostas usando IA para consultar dados de arquivos Excel, com uma interface HTML amigável para interação.

## 🎯 Funcionalidades:
- Carregamento e processamento de dados Excel
- Criação de embeddings com modelo multilíngue
- Armazenamento em banco de dados vetorial
- Sistema de Q&A com HuggingFace (gratuito)
- **Interface HTML interativa e amigável**

---

## 1. Instalação das Dependências

In [ ]:
# Instalar dependências necessárias
!pip install pandas sentence-transformers chromadb langchain langchain-community openpyxl transformers torch --quiet

print("✅ Dependências instaladas com sucesso!")

## 2. Importação das Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import json
import base64
from IPython.display import HTML, display
import threading
import time
warnings.filterwarnings('ignore')

# LangChain imports
from langchain.schema import Document
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma

# Transformers para Q&A
from transformers import pipeline
import torch

print("✅ Bibliotecas importadas com sucesso!")

## 3. Configurações Globais

In [ ]:
# Configurações do sistema
CONFIG = {
    'embedding_model': 'BAAI/bge-m3',  # Modelo multilíngue otimizado
    'qa_model': 'distilbert-base-cased-distilled-squad',  # Modelo Q&A
    'chunk_size': 1000,               # Tamanho dos chunks de texto
    'k_documents': 5,                 # Número de documentos a recuperar
    'persist_directory': 'chromadb',  # Diretório do banco vetorial
    'sample_fraction': 0.5            # Fração dos dados a processar
}

print("⚙️ Configurações definidas:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 4. Carregamento e Análise dos Dados

In [ ]:
def load_and_analyze_excel(file_path):
    """
    Carrega e analisa um arquivo Excel.
    """
    try:
        # Carregamento do arquivo
        data = pd.read_excel(file_path)
        
        print(f"📊 Arquivo carregado: {file_path}")
        print(f"📏 Dimensões originais: {data.shape}")
        
        # Análise básica
        print(f"🔍 Colunas encontradas: {list(data.columns)[:5]}...")  # Primeiras 5 colunas
        print(f"📈 Linhas com dados: {data.dropna(how='all').shape[0]}")
        
        return data
        
    except Exception as e:
        print(f"❌ Erro ao carregar arquivo: {e}")
        return None

# Carregamento dos dados (ajuste o caminho conforme necessário)
file_path = "dados.xlsx"  # Substitua pelo caminho do seu arquivo
raw_data = load_and_analyze_excel(file_path)

## 5. Limpeza e Preparação dos Dados

In [ ]:
def clean_and_prepare_data(data, sample_fraction=0.5):
    """
    Limpa e prepara os dados para processamento.
    """
    if data is None:
        return None
    
    # Cópia dos dados
    clean_data = data.copy()
    
    # Remove linhas completamente vazias
    clean_data = clean_data.dropna(how='all')
    
    # Amostragem dos dados
    if sample_fraction < 1.0:
        sample_size = int(len(clean_data) * sample_fraction)
        clean_data = clean_data.head(sample_size)
        print(f"📊 Usando {sample_fraction*100}% dos dados: {sample_size} linhas")
    
    # Colunas a remover (ajuste conforme necessário)
    columns_to_drop = [
        'Ramal', 'Contador', 'Tipo de SS', 'Ret.Operacao', 
        'Vandalismo', 'Criticidade', 'Tipo Solicit'
    ]
    
    # Remove colunas desnecessárias
    clean_data = clean_data.drop(columns=columns_to_drop, errors='ignore')
    
    # Converte colunas para string
    for col in clean_data.columns:
        clean_data[col] = clean_data[col].astype(str)
    
    print(f"✅ Dados limpos - Dimensões finais: {clean_data.shape}")
    
    return clean_data

# Limpeza dos dados
processed_data = clean_and_prepare_data(raw_data, CONFIG['sample_fraction'])

# Visualização dos primeiros registros
if processed_data is not None:
    print("\n🔍 Primeiros 3 registros processados:")
    display(processed_data.head(3))

## 6. Criação de Documentos para Embedding

In [ ]:
def create_documents(data):
    """
    Converte DataFrame em documentos LangChain.
    """
    if data is None:
        return []
    
    documents = []
    
    for idx, row in data.iterrows():
        # Cria texto estruturado a partir da linha
        text_parts = []
        for col in data.columns:
            value = row[col]
            if pd.notna(value) and str(value).strip() and str(value) != 'nan':
                text_parts.append(f"{col}: {value}")
        
        if text_parts:  # Só adiciona se houver conteúdo
            text = " | ".join(text_parts)
            
            # Cria documento com metadados
            doc = Document(
                page_content=text,
                metadata={
                    "row_index": idx,
                    "source": "excel_data",
                    "num_fields": len(text_parts)
                }
            )
            documents.append(doc)
    
    print(f"📄 Criados {len(documents)} documentos para embedding")
    
    return documents

# Criação dos documentos
documents = create_documents(processed_data)

## 7. Configuração do Modelo de Embeddings

In [ ]:
def setup_embeddings(model_name):
    """
    Configura o modelo de embeddings.
    """
    try:
        print(f"🤖 Carregando modelo de embeddings: {model_name}")
        
        embeddings = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={'device': 'cpu'},
            encode_kwargs={'normalize_embeddings': True}
        )
        
        print("✅ Modelo de embeddings carregado com sucesso!")
        return embeddings
        
    except Exception as e:
        print(f"❌ Erro ao carregar modelo de embeddings: {e}")
        return None

# Configuração do modelo de embeddings
embeddings = setup_embeddings(CONFIG['embedding_model'])

## 8. Criação do Banco de Dados Vetorial

In [ ]:
def create_vector_database(documents, embeddings, persist_dir):
    """
    Cria o banco de dados vetorial ChromaDB.
    """
    if not documents or embeddings is None:
        print("❌ Documentos ou embeddings não disponíveis")
        return None
    
    try:
        print(f"🗄️ Criando banco de dados vetorial em: {persist_dir}")
        print(f"📊 Processando {len(documents)} documentos...")
        print("⏳ Esta operação pode demorar alguns minutos...")
        
        # Cria o banco vetorial
        vectordb = Chroma.from_documents(
            documents=documents,
            embedding=embeddings,
            persist_directory=persist_dir
        )
        
        print("✅ Banco de dados vetorial criado com sucesso!")
        print(f"📈 Total de vetores armazenados: {vectordb._collection.count()}")
        
        return vectordb
        
    except Exception as e:
        print(f"❌ Erro ao criar banco vetorial: {e}")
        return None

# Criação do banco vetorial
vectordb = create_vector_database(documents, embeddings, CONFIG['persist_directory'])

## 9. Configuração do Sistema Q&A com HuggingFace

In [ ]:
def setup_qa_pipeline():
    """
    Configura o pipeline de Q&A do HuggingFace.
    """
    try:
        print(f"🤖 Carregando modelo Q&A: {CONFIG['qa_model']}")
        
        # Pipeline de Q&A otimizado
        qa_pipeline = pipeline(
            "question-answering",
            model=CONFIG['qa_model'],
            device=-1  # CPU
        )
        
        print("✅ Pipeline Q&A configurado com sucesso!")
        return qa_pipeline
        
    except Exception as e:
        print(f"❌ Erro ao configurar pipeline Q&A: {e}")
        return None

# Configuração do pipeline Q&A
qa_pipeline = setup_qa_pipeline()

## 10. Sistema de Q&A Integrado

In [ ]:
def ask_question_ai(question, k=5):
    """
    Faz uma pergunta ao sistema de Q&A integrado.
    """
    try:
        # Verificar se o sistema está disponível
        if qa_pipeline is None or vectordb is None:
            return {
                "error": "Sistema não configurado",
                "answer": "Sistema Q&A não está disponível. Execute as células anteriores."
            }
        
        # Buscar documentos relevantes
        docs = vectordb.similarity_search(question, k=k)
        
        if not docs:
            return {
                "answer": "Não encontrei informações relevantes para sua pergunta.",
                "confidence": 0.0,
                "sources": []
            }
        
        # Preparar contexto
        context = " ".join([doc.page_content for doc in docs])
        context = context[:2000]  # Limitar tamanho
        
        # Usar pipeline Q&A
        result = qa_pipeline(question=question, context=context)
        
        # Estruturar resposta
        response = {
            "answer": result['answer'],
            "confidence": round(result['score'] * 100, 1),
            "sources": [doc.page_content[:200] + "..." for doc in docs[:3]],
            "num_sources": len(docs)
        }
        
        return response
        
    except Exception as e:
        return {
            "error": str(e),
            "answer": f"Erro ao processar pergunta: {e}"
        }

print("✅ Sistema Q&A integrado configurado!")

## 11. Interface HTML Interativa

In [1]:
# Variável global para armazenar histórico de conversas
conversation_history = []

def create_html_interface():
    """
    Cria interface HTML interativa para o sistema Q&A.
    """
    html_code = """
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Sistema Q&A com IA</title>
        <style>
            * {
                margin: 0;
                padding: 0;
                box-sizing: border-box;
            }
            
            body {
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
                background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                min-height: 100vh;
                padding: 20px;
            }
            
            .container {
                max-width: 1000px;
                margin: 0 auto;
                background: white;
                border-radius: 20px;
                box-shadow: 0 20px 40px rgba(0,0,0,0.1);
                overflow: hidden;
            }
            
            .header {
                background: linear-gradient(135deg, #4facfe 0%, #00f2fe 100%);
                color: white;
                padding: 30px;
                text-align: center;
            }
            
            .header h1 {
                font-size: 2.5em;
                margin-bottom: 10px;
                text-shadow: 0 2px 4px rgba(0,0,0,0.3);
            }
            
            .header p {
                font-size: 1.2em;
                opacity: 0.9;
            }
            
            .chat-container {
                height: 500px;
                overflow-y: auto;
                padding: 20px;
                background: #f8f9fa;
                border-bottom: 1px solid #e9ecef;
            }
            
            .message {
                margin-bottom: 20px;
                animation: fadeIn 0.5s ease-in;
            }
            
            .user-message {
                text-align: right;
            }
            
            .user-message .bubble {
                background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                color: white;
                display: inline-block;
                padding: 15px 20px;
                border-radius: 20px 20px 5px 20px;
                max-width: 70%;
                word-wrap: break-word;
                box-shadow: 0 4px 8px rgba(0,0,0,0.1);
            }
            
            .ai-message .bubble {
                background: white;
                color: #333;
                display: inline-block;
                padding: 15px 20px;
                border-radius: 20px 20px 20px 5px;
                max-width: 70%;
                word-wrap: break-word;
                border: 2px solid #e9ecef;
                box-shadow: 0 4px 8px rgba(0,0,0,0.1);
            }
            
            .confidence {
                font-size: 0.9em;
                color: #666;
                margin-top: 5px;
            }
            
            .sources {
                font-size: 0.8em;
                color: #888;
                margin-top: 10px;
                padding-top: 10px;
                border-top: 1px solid #eee;
            }
            
            .input-container {
                padding: 20px;
                background: white;
                display: flex;
                gap: 10px;
            }
            
            .input-field {
                flex: 1;
                padding: 15px 20px;
                border: 2px solid #e9ecef;
                border-radius: 25px;
                font-size: 16px;
                outline: none;
                transition: all 0.3s ease;
            }
            
            .input-field:focus {
                border-color: #4facfe;
                box-shadow: 0 0 0 3px rgba(79, 172, 254, 0.1);
            }
            
            .send-button {
                background: linear-gradient(135deg, #4facfe 0%, #00f2fe 100%);
                color: white;
                border: none;
                padding: 15px 25px;
                border-radius: 25px;
                cursor: pointer;
                font-size: 16px;
                font-weight: bold;
                transition: all 0.3s ease;
                box-shadow: 0 4px 8px rgba(0,0,0,0.1);
            }
            
            .send-button:hover {
                transform: translateY(-2px);
                box-shadow: 0 6px 12px rgba(0,0,0,0.15);
            }
            
            .send-button:disabled {
                opacity: 0.6;
                cursor: not-allowed;
                transform: none;
            }
            
            .loading {
                display: none;
                text-align: center;
                padding: 20px;
                color: #666;
            }
            
            .loading-dots {
                display: inline-block;
                animation: loading 1.5s infinite;
            }
            
            .examples {
                padding: 20px;
                background: #f8f9fa;
                border-top: 1px solid #e9ecef;
            }
            
            .examples h3 {
                margin-bottom: 15px;
                color: #333;
            }
            
            .example-button {
                background: white;
                border: 1px solid #ddd;
                padding: 10px 15px;
                margin: 5px;
                border-radius: 15px;
                cursor: pointer;
                transition: all 0.3s ease;
                display: inline-block;
                font-size: 14px;
            }
            
            .example-button:hover {
                background: #4facfe;
                color: white;
                border-color: #4facfe;
            }
            
            @keyframes fadeIn {
                from { opacity: 0; transform: translateY(20px); }
                to { opacity: 1; transform: translateY(0); }
            }
            
            @keyframes loading {
                0%, 20% { transform: scale(1); }
                50% { transform: scale(1.2); }
                80%, 100% { transform: scale(1); }
            }
            
            .status-indicator {
                position: absolute;
                top: 20px;
                right: 20px;
                padding: 8px 15px;
                border-radius: 20px;
                font-size: 12px;
                font-weight: bold;
            }
            
            .status-online {
                background: #28a745;
                color: white;
            }
            
            .status-offline {
                background: #dc3545;
                color: white;
            }
        </style>
    </head>
    <body>
        <div class="container">
            <div class="header">
                <div class="status-indicator status-online" id="status">🟢 Sistema Online</div>
                <h1>🤖 Assistente IA para Dados Excel</h1>
                <p>Faça perguntas sobre seus dados e obtenha respostas inteligentes</p>
            </div>
            
            <div class="chat-container" id="chatContainer">
                <div class="message ai-message">
                    <div class="bubble">
                        👋 Olá! Sou seu assistente de IA especializado em análise de dados Excel. 
                        Posso responder perguntas sobre solicitações, estações, circuitos e muito mais!
                        <br><br>
                        💡 <strong>Dica:</strong> Seja específico em suas perguntas para obter melhores respostas.
                    </div>
                </div>
            </div>
            
            <div class="loading" id="loading">
                <span class="loading-dots">🤖 Processando sua pergunta</span>
            </div>
            
            <div class="input-container">
                <input type="text" class="input-field" id="questionInput" 
                       placeholder="Digite sua pergunta aqui..." 
                       onkeypress="handleKeyPress(event)">
                <button class="send-button" id="sendButton" onclick="sendQuestion()">Enviar</button>
            </div>
            
            <div class="examples">
                <h3>💡 Exemplos de perguntas:</h3>
                <div class="example-button" onclick="setQuestion('Quantas solicitações existem para a Estação Cavaleiro?')">Solicitações por estação</div>
                <div class="example-button" onclick="setQuestion('Quais são os diferentes tipos de circuitos de via?')">Tipos de circuitos</div>
                <div class="example-button" onclick="setQuestion('Qual é o MTTF de SINCDVROD?')">MTTF específico</div>
                <div class="example-button" onclick="setQuestion('Mostre problemas na estação Shopping')">Problemas por local</div>
                <div class="example-button" onclick="setQuestion('Qual estação tem mais problemas reportados?')">Análise comparativa</div>
            </div>
        </div>
        
        <script>
            function setQuestion(question) {
                document.getElementById('questionInput').value = question;
                document.getElementById('questionInput').focus();
            }
            
            function handleKeyPress(event) {
                if (event.key === 'Enter') {
                    sendQuestion();
                }
            }
            
            function sendQuestion() {
                const input = document.getElementById('questionInput');
                const question = input.value.trim();
                
                if (!question) {
                    alert('Por favor, digite uma pergunta!');
                    return;
                }
                
                // Adicionar pergunta do usuário
                addMessage(question, 'user');
                
                // Limpar input e mostrar loading
                input.value = '';
                showLoading(true);
                
                // Simular processamento (em implementação real, chamaria a função Python)
                setTimeout(() => {
                    // Esta parte será conectada com o backend Python
                    processQuestion(question);
                }, 1000);
            }
            
            function addMessage(content, type, confidence = null, sources = null) {
                const chatContainer = document.getElementById('chatContainer');
                const messageDiv = document.createElement('div');
                messageDiv.className = `message ${type}-message`;
                
                let messageContent = `<div class="bubble">${content}`;
                
                if (confidence !== null) {
                    messageContent += `<div class="confidence">📊 Confiança: ${confidence}%</div>`;
                }
                
                if (sources && sources.length > 0) {
                    messageContent += `<div class="sources">📚 Fontes consultadas: ${sources.length} documentos</div>`;
                }
                
                messageContent += '</div>';
                messageDiv.innerHTML = messageContent;
                
                chatContainer.appendChild(messageDiv);
                chatContainer.scrollTop = chatContainer.scrollHeight;
            }
            
            function showLoading(show) {
                const loading = document.getElementById('loading');
                const sendButton = document.getElementById('sendButton');
                
                loading.style.display = show ? 'block' : 'none';
                sendButton.disabled = show;
                
                if (show) {
                    const chatContainer = document.getElementById('chatContainer');
                    chatContainer.scrollTop = chatContainer.scrollHeight;
                }
            }
            
            function processQuestion(question) {
                // Esta função será conectada com o sistema Python
                // Por enquanto, simula uma resposta
                
                // Chamar função Python através do Jupyter
                const code = `
                    result = ask_question_ai("${question}")
                    import json
                    print("RESULT_JSON:" + json.dumps(result))
                `;
                
                // Executar código Python (isso funcionará no Jupyter)
                if (typeof Jupyter !== 'undefined') {
                    Jupyter.notebook.kernel.execute(code, {
                        iopub: {
                            output: function(msg) {
                                if (msg.content && msg.content.text) {
                                    const text = msg.content.text;
                                    if (text.includes('RESULT_JSON:')) {
                                        const jsonStr = text.split('RESULT_JSON:')[1];
                                        try {
                                            const result = JSON.parse(jsonStr);
                                            displayResult(result);
                                        } catch (e) {
                                            displayError('Erro ao processar resposta');
                                        }
                                    }
                                }
                            }
                        }
                    });
                } else {
                    // Fallback para demonstração
                    setTimeout(() => {
                        const demoResult = {
                            answer: "Esta é uma resposta de demonstração. Para funcionalidade completa, execute no Jupyter Notebook.",
                            confidence: 85.5,
                            sources: ["Documento 1", "Documento 2"]
                        };
                        displayResult(demoResult);
                    }, 2000);
                }
            }
            
            function displayResult(result) {
                showLoading(false);
                
                if (result.error) {
                    addMessage(`❌ ${result.answer}`, 'ai');
                } else {
                    addMessage(result.answer, 'ai', result.confidence, result.sources);
                }
            }
            
            function displayError(message) {
                showLoading(false);
                addMessage(`❌ ${message}`, 'ai');
            }
            
            // Verificar status do sistema
            function checkSystemStatus() {
                // Esta função verificaria se o sistema Python está funcionando
                const status = document.getElementById('status');
                
                if (typeof ask_question_ai !== 'undefined') {
                    status.textContent = '🟢 Sistema Online';
                    status.className = 'status-indicator status-online';
                } else {
                    status.textContent = '🔴 Sistema Offline';
                    status.className = 'status-indicator status-offline';
                }
            }
            
            // Verificar status ao carregar
            setTimeout(checkSystemStatus, 1000);
        </script>
    </body>
    </html>
    """
    
    return html_code

# Função para exibir a interface
def show_interface():
    """
    Exibe a interface HTML no Jupyter Notebook.
    """
    html_code = create_html_interface()
    display(HTML(html_code))

print("✅ Interface HTML criada com sucesso!")
print("Execute show_interface() para exibir a interface interativa.")

✅ Interface HTML criada com sucesso!
Execute show_interface() para exibir a interface interativa.


## 12. Exibir Interface Interativa

In [ ]:
# Exibir a interface HTML interativa
show_interface()

## 13. Testes e Exemplos

In [ ]:
# Função para testar o sistema via código
def test_qa_system():
    """
    Testa o sistema Q&A com perguntas de exemplo.
    """
    print("🧪 TESTANDO SISTEMA Q&A")
    print("="*50)
    
    test_questions = [
        "Quantas solicitações existem para a Estação Cavaleiro?",
        "Quais são os diferentes tipos de circuitos de via?",
        "Qual é o MTTF de SINCDVROD?",
        "Qual estação tem mais problemas reportados?"
    ]
    
    for i, question in enumerate(test_questions, 1):
        print(f"\n--- TESTE {i} ---")
        print(f"❓ Pergunta: {question}")
        
        result = ask_question_ai(question)
        
        print(f"🤖 Resposta: {result['answer']}")
        if 'confidence' in result:
            print(f"📊 Confiança: {result['confidence']}%")
        if 'num_sources' in result:
            print(f"📚 Fontes: {result['num_sources']} documentos")
        
        print("-" * 50)

# Executar testes
if vectordb is not None and qa_pipeline is not None:
    test_qa_system()
else:
    print("⚠️ Sistema não está completamente configurado. Execute as células anteriores.")

## 14. Instruções de Uso

In [ ]:
print("📋 INSTRUÇÕES DE USO DO SISTEMA Q&A")
print("="*60)

print("\n🎯 INTERFACE HTML INTERATIVA:")
print("- Use a interface acima para fazer perguntas")
print("- Digite sua pergunta e clique em 'Enviar'")
print("- Ou clique nos exemplos para testar rapidamente")

print("\n💻 VIA CÓDIGO PYTHON:")
print("- ask_question_ai('sua pergunta')")
print("- test_qa_system()  # Para executar testes")

print("\n💡 DICAS PARA MELHORES RESULTADOS:")
print("- Seja específico em suas perguntas")
print("- Use termos que aparecem nos dados (ex: nomes de estações)")
print("- Pergunte sobre quantidades, tipos, problemas, etc.")

print("\n🔧 CONFIGURAÇÕES:")
print(f"- Modelo de embeddings: {CONFIG['embedding_model']}")
print(f"- Modelo Q&A: {CONFIG['qa_model']}")
print(f"- Documentos por consulta: {CONFIG['k_documents']}")
print(f"- Dados processados: {CONFIG['sample_fraction']*100}%")

print("\n✅ SISTEMA PRONTO PARA USO!")
print("="*60)

## 15. Backup e Exportação

In [ ]:
import shutil
from datetime import datetime

def backup_system():
    """
    Cria backup do sistema completo.
    """
    try:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup_name = f"qa_system_backup_{timestamp}"
        
        # Backup do banco vetorial
        if CONFIG['persist_directory'] and Path(CONFIG['persist_directory']).exists():
            shutil.make_archive(backup_name, 'zip', CONFIG['persist_directory'])
            print(f"✅ Backup criado: {backup_name}.zip")
            
            # Para Google Colab - fazer download
            try:
                from google.colab import files
                files.download(f"{backup_name}.zip")
                print("📥 Download do backup iniciado")
            except ImportError:
                print("💾 Backup salvo localmente")
        else:
            print("❌ Diretório do banco vetorial não encontrado")
            
    except Exception as e:
        print(f"❌ Erro ao criar backup: {e}")

def export_conversation_history():
    """
    Exporta histórico de conversas.
    """
    try:
        if conversation_history:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"conversation_history_{timestamp}.json"
            
            with open(filename, 'w', encoding='utf-8') as f:
                json.dump(conversation_history, f, ensure_ascii=False, indent=2)
            
            print(f"✅ Histórico exportado: {filename}")
            
            # Para Google Colab
            try:
                from google.colab import files
                files.download(filename)
            except ImportError:
                pass
        else:
            print("📝 Nenhum histórico de conversa disponível")
            
    except Exception as e:
        print(f"❌ Erro ao exportar histórico: {e}")

print("💾 Funções de backup configuradas:")
print("- backup_system()  # Backup do banco vetorial")
print("- export_conversation_history()  # Exportar conversas")